## 395 Final Project: Predicting Misc Genres
-CNN implementation on images

In [3]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

SEED = 4


### Load Data
- split training set into train and validation (15% of overall training set)
- define genres numerically (0 to 9)

In [4]:
#load metadata table
metadata = pd.read_csv("./music_data/metadata.csv")

#get training data
train_rows = metadata[metadata["split"] == "train"]
unique_train_songs = train_rows[["song", "genre"]].drop_duplicates()

#split 15% of training songs into validation set
_, val_songs = train_test_split(unique_train_songs, test_size=0.15, stratify=unique_train_songs["genre"], random_state=SEED)

#update labels for songs in validation set
val_song_names = set(val_songs["song"])
new_splits = []
for _, row in metadata.iterrows():
    if row["song"] in val_song_names:
        new_splits.append("val")
    else:
        new_splits.append(row["split"])
metadata["split"] = new_splits

#define numerical labels for genres
class_names = sorted(metadata["genre"].unique())

genre_to_num = {}
for i, genre in enumerate(class_names):
    genre_to_num[genre] = i

labels = []
for genre in metadata["genre"]:
    labels.append(genre_to_num[genre])
metadata["label"] = labels

#load .npy files into arrays to CNN


In [5]:
#load .npy files into arrays
def load_split(split):
    X_list = []
    y_list = []

    #subset of data in either "train", "val", or "test"
    subset = metadata[metadata["split"] == split]

    #load spectrogram in row and save label
    for _, row in subset.iterrows():
        spectrogram = np.load(row["spectrogram_path"]) #shape = (128, 130)
        X_list.append(spectrogram)
        y_list.append(row["label"])

    X = np.array(X_list, dtype=np.float32) #shape (N, 128, 130)
    y = np.array(y_list, dtype=np.int64) #shape (N,)

    X = X.reshape(-1, 128, 130, 1)

    return X, y

X_train, y_train = load_split("train")
X_val, y_val = load_split("val")
X_test, y_test = load_split("test")

print("Genres:", class_names)
print("\nX_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("\nX_val shape:", X_val.shape)
print("y_val shape:", y_val.shape)
print("\nX_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

print("Pixel value range:", X_train.min(), "-", X_train.max())

Genres: ['blues', 'classical', 'country', 'disco', 'hiphop', 'jazz', 'metal', 'pop', 'reggae', 'rock']

X_train shape: (6787, 128, 130, 1)
y_train shape: (6787,)

X_val shape: (1194, 128, 130, 1)
y_val shape: (1194,)

X_test shape: (1999, 128, 130, 1)
y_test shape: (1999,)
Pixel value range: 0.0 - 1.0
